In [5]:
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

import sys
sys.path.append("..")
from src.data.preprocessing import build_leave_one_out
from src.metrics.ranking import compute_rank, Metrics
from src.models.baseline_popularity import PopularityBaseline
from src.models.baseline_als import ALSBaseline

In [6]:
PROCESSED_DIR = Path("../data/processed")
K = 5

In [7]:
with open(PROCESSED_DIR / "sequences.pkl", "rb") as f:
    sequences = pickle.load(f)

with open(PROCESSED_DIR / "item_mappings.pkl", "rb") as f:
    mappings = pickle.load(f)
vocab_size = mappings["vocab_size"]

print(f"Загружено {len(sequences)} пользователей, vocab_size = {vocab_size}")

Загружено 197519 пользователей, vocab_size = 47975


In [10]:
split_data = build_leave_one_out(sequences)
len(split_data)

197519

In [11]:
train_prefixes = {uid: data["train"] for uid, data in split_data.items()}

In [18]:
popularity = PopularityBaseline()
popularity.fit(train_prefixes, vocab_size)

In [14]:
als = ALSBaseline(factors=64, regularization=0.01, iterations=20, alpha=40)
als.fit(train_prefixes, vocab_size)

100%|██████████| 20/20 [01:24<00:00,  4.25s/it]


In [21]:
def evaluate_model(model, target_key, k=K):
    recalls = []
    ndcgs = []
    mrrs = []

    for user_id, data in tqdm(split_data.items(), desc="Evaluating"):
        true_idx = data[target_key]

        if isinstance(model, PopularityBaseline):
            scores = model.score(session=None)
        elif isinstance(model, ALSBaseline):
            user_idx = model.user2idx[user_id]
            scores = model.score(user_idx)

        rank = compute_rank(scores, true_idx)
        recalls.append(Metrics.recall_at_k(rank, k))
        ndcgs.append(Metrics.ndcg_at_k(rank, k))
        mrrs.append(Metrics.mrr(rank))

    return {
        f"Recall@{k}": np.mean(recalls),
        f"NDCG@{k}": np.mean(ndcgs),
        "MRR": np.mean(mrrs)
    }

In [22]:
from tqdm import tqdm

In [23]:
print("\nОценка на валидационных целях (val_target):")

pop_val = evaluate_model(popularity, "val_target", k=K)
als_val = evaluate_model(als, "val_target", k=K)

print("Popularity (val):", pop_val)
print("ALS (val):", als_val)


Оценка на валидационных целях (val_target):


Evaluating: 100%|██████████| 197519/197519 [08:45<00:00, 376.01it/s]

Popularity (val): {'Recall@5': np.float64(0.034356188518572896), 'NDCG@5': np.float64(0.02186997548404705), 'MRR': np.float64(0.025508439312613197)}
ALS (val): {'Recall@5': np.float64(0.054106187252871876), 'NDCG@5': np.float64(0.03464388712061111), 'MRR': np.float64(0.042885866625186043)}


In [24]:
print("\nОценка на тестовых целях (test_target):")

pop_test = evaluate_model(popularity, "test_target", k=K)
als_test = evaluate_model(als, "test_target", k=K)

print("Popularity (test):", pop_test)
print("ALS (test):", als_test)


Оценка на тестовых целях (test_target):


Evaluating: 100%|██████████| 197519/197519 [08:49<00:00, 372.92it/s]

Popularity (test): {'Recall@5': np.float64(0.03161214870468158), 'NDCG@5': np.float64(0.019986848013186502), 'MRR': np.float64(0.023435925742661783)}
ALS (test): {'Recall@5': np.float64(0.04722583650180489), 'NDCG@5': np.float64(0.030142640228151822), 'MRR': np.float64(0.03801560968850304)}


In [26]:
results = pd.DataFrame({
    "Модель": ["Popularity", "ALS"],
    "Recall@5 (val)": [pop_val["Recall@5"], als_val["Recall@5"]],
    "Recall@5 (test)": [pop_test["Recall@5"], als_test["Recall@5"]],
    "NDCG@5 (val)": [pop_val["NDCG@5"], als_val["NDCG@5"]],
    "NDCG@5 (test)": [pop_test["NDCG@5"], als_test["NDCG@5"]],
    "MRR (val)": [pop_val["MRR"], als_val["MRR"]],
    "MRR (test)": [pop_test["MRR"], als_test["MRR"]],
})

results

,Модель,Recall@5 (val),Recall@5 (test),NDCG@5 (val),NDCG@5 (test),MRR (val),MRR (test)
0,Popularity,0.034356,0.031612,0.021870,0.019987,0.025508,0.023436
1,ALS,0.054106,0.047226,0.034644,0.030143,0.042886,0.038016
